# Vue2 + Vue3 框架 · 第 07 天

## Vuex 状态管理与购物车模块

> 本篇依据第 07 天 MD 笔记、PPT 和随堂代码整理。重点是多个组件共享数据时，如何用 Vuex 集中管理状态，并把购物车案例拆成模块。

## 一、为什么需要 Vuex

当多个组件需要读取或修改同一份数据时，如果层层通过 props 和 `$emit` 传递，会出现链路长、维护困难的问题。Vuex 提供一个集中式 store：

```text
组件 → dispatch(action) → commit(mutation) → state
组件 ← 读取 state / getter ← store
```

适合用户信息、购物车、权限、主题等跨页面共享状态。局部输入框、只在单个组件使用的数据仍然放在组件自己的 `data` 中。

Vue 3 项目通常使用 Pinia；本天使用 Vue 2 生态中的 Vuex 3/4 思路。

## 二、创建 Vuex 仓库

```bash
npm install vuex@3
```

```js
// src/store/index.js
import Vue from 'vue'
import Vuex from 'vuex'

Vue.use(Vuex)

export default new Vuex.Store({
  state: {
    count: 0,
    user: null
  },
  mutations: {},
  actions: {},
  getters: {}
})
```

在入口文件注入：

```js
import store from './store'
new Vue({ render: h => h(App), store }).$mount('#app')
```

注入后，所有组件都可以通过 `this.$store` 访问仓库。

## 三、核心概念：state

### 3.1 读取 state

```js
this.$store.state.count
```

模板中：

```vue
<p>当前数量：{{ $store.state.count }}</p>
```

更推荐通过计算属性集中读取：

```js
computed: {
  count () {
    return this.$store.state.count
  }
}
```

### 3.2 `mapState`

```js
import { mapState } from 'vuex'
export default {
  computed: {
    ...mapState(['count', 'user'])
  }
}
```

也可以使用对象映射解决命名冲突：

```js
computed: {
  ...mapState({ total: state => state.count })
}
```

## 四、严格模式与单向数据流

不能在组件中直接修改 `state`：

```js
this.$store.state.count++ // 错误
```

开启严格模式后，Vuex 会在控制台提示非法修改：

```js
export default new Vuex.Store({
  strict: process.env.NODE_ENV !== 'production',
  state: { count: 0 }
})
```

正确流程是提交 mutation，由 mutation 修改 state：

```text
组件不能直接改 state
组件 commit mutation
mutation 同步修改 state
```

这样所有状态变化都有明确入口，调试工具也能记录每次变更。

## 五、mutations：同步修改 state

```js
const store = new Vuex.Store({
  state: { count: 0 },
  mutations: {
    increment (state) {
      state.count++
    },
    decrement (state) {
      state.count--
    },
    setCount (state, value) {
      state.count = value
    }
  }
})
```

组件中提交：

```js
this.$store.commit('increment')
this.$store.commit('setCount', 10)
```

对象风格提交：

```js
this.$store.commit({ type: 'setCount', value: 10 })
```

mutation 必须是同步函数，这样 devtools 才能准确记录修改前后的状态。

### `mapMutations`

```js
import { mapMutations } from 'vuex'
export default {
  methods: {
    ...mapMutations(['increment', 'decrement'])
  }
}
```

## 六、actions：异步业务逻辑

异步请求、定时器等放在 actions 中，完成后再提交 mutation：

```js
const store = new Vuex.Store({
  state: { list: [] },
  mutations: {
    setList (state, list) { state.list = list }
  },
  actions: {
    async getList ({ commit }) {
      const res = await fetch('/api/list')
      const data = await res.json()
      commit('setList', data)
    }
  }
})
```

组件通过 `dispatch` 调用：

```js
this.$store.dispatch('getList')
```

带参数：

```js
this.$store.dispatch('getList', { page: 1, size: 10 })
```

`mapActions`：

```js
import { mapActions } from 'vuex'
methods: { ...mapActions(['getList']) }
```

## 七、getters：派生数据

Getter 类似计算属性，适合对 state 做过滤、统计和格式化：

```js
getters: {
  cartTotal (state) {
    return state.cart.reduce((sum, item) => sum + item.price * item.count, 0)
  },
  checkedList (state) {
    return state.cart.filter(item => item.checked)
  }
}
```

读取：

```js
this.$store.getters.cartTotal
```

辅助函数：

```js
import { mapGetters } from 'vuex'
computed: { ...mapGetters(['cartTotal', 'checkedList']) }
```

Getter 只负责计算结果，不要在 getter 中修改 state 或发送请求。

## 八、Vuex module 模块化

单一 store 变大后，可以按业务拆分模块：

```js
const user = {
  namespaced: true,
  state: () => ({ token: '' }),
  mutations: {
    setToken (state, token) { state.token = token }
  },
  actions: {
    logout ({ commit }) { commit('setToken', '') }
  },
  getters: {
    isLogin: state => Boolean(state.token)
  }
}

export default new Vuex.Store({
  modules: { user }
})
```

模块中：

```js
this.$store.state.user.token
this.$store.getters['user/isLogin']
this.$store.commit('user/setToken', token)
this.$store.dispatch('user/logout')
```

配合辅助函数时指定命名空间：

```js
import { createNamespacedHelpers } from 'vuex'
const { mapState, mapActions } = createNamespacedHelpers('user')
```

## 九、综合案例：Vuex 购物车

### 9.1 状态设计

```js
state: {
  cartList: [
    { id: 1, name: '苹果', price: 6, count: 2, isChecked: true }
  ]
}
```

建议把购物车单独放在 `store/modules/cart.js`，由它负责添加商品、修改数量、删除和全选。

### 9.2 典型 mutations

```js
mutations: {
  setCartList (state, list) { state.cartList = list },
  changeCount (state, { id, count }) {
    const item = state.cartList.find(item => item.id === id)
    if (item) item.count = Math.max(1, count)
  },
  removeCart (state, id) {
    state.cartList = state.cartList.filter(item => item.id !== id)
  },
  toggleAll (state, checked) {
    state.cartList.forEach(item => { item.isChecked = checked })
  }
}
```

### 9.3 典型 getters

```js
getters: {
  isAllChecked: state => state.cartList.length > 0 && state.cartList.every(item => item.isChecked),
  checkedCount: state => state.cartList
    .filter(item => item.isChecked)
    .reduce((sum, item) => sum + item.count, 0),
  checkedTotal: state => state.cartList
    .filter(item => item.isChecked)
    .reduce((sum, item) => sum + item.price * item.count, 0)
}
```

组件通过 `mapState`、`mapGetters` 渲染，通过 `mapMutations` 或 `mapActions` 触发修改。组件不直接操作 Vuex state。

## 十、输入框与 Vuex 双向绑定

不要直接把 `v-model` 绑定到 `this.$store.state`。使用 getter + setter 或显式事件：

```vue
<input :value="count" @input="updateCount">
```

```js
computed: {
  count () { return this.$store.state.count }
},
methods: {
  updateCount (event) {
    this.$store.commit('setCount', Number(event.target.value))
  }
}
```

这样数据写入仍然经过 mutation，严格模式不会报错。

## 十一、当天速记

- `state` 存共享数据，`getters` 计算派生结果。
- `mutations` 同步修改 state，`actions` 处理异步并 dispatch/commit。
- 组件不能直接修改 state，状态变化必须经过明确入口。
- `mapState`、`mapGetters`、`mapMutations`、`mapActions` 减少重复代码。
- 业务复杂时使用 namespaced module 拆分用户、购物车等领域。
- 购物车的总价、全选和选中数量适合 getters；数量修改、删除和全选适合 mutations。